# Imports

In [1]:
import torch
import torch.nn as nn
from transformers import GPT2Tokenizer
import os

# ============================================
# ENVIRONMENT SETUP FOR KAGGLE T4 x2
# ============================================

# Disable warnings and set optimal settings
os.environ["TOKENIZERS_PARALLELISM"] = "false"
torch.backends.cudnn.benchmark = True  # Optimize for T4 architecture

# GPU detection and configuration
print("=" * 60)
print("KAGGLE T4 x2 GPU CONFIGURATION")
print("=" * 60)

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Number of GPUs detected: {torch.cuda.device_count()}")

# Display GPU info
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"\nGPU {i}: {props.name}")
    print(f"  Memory: {props.total_memory / 1e9:.1f} GB")
    print(f"  Compute Capability: {props.major}.{props.minor}")

# Set primary device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n✓ Primary device: {device}")
print("=" * 60)

# ============================================
# MODEL CONFIGURATION
# ============================================

GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": True,
    "use_float16": True,  # Added for T4 optimization (T4 doesn't support bfloat16)
}

print("\nModel Configuration:")
print(GPT_CONFIG_124M)

# ============================================
# TOKENIZER (Fixed for Kaggle - no tiktoken network issues)
# ============================================

print("\nLoading tokenizer...")
try:
    # Using Hugging Face transformers instead of tiktoken
    tokeniser = GPT2Tokenizer.from_pretrained('gpt2')
    tokeniser.pad_token = tokeniser.eos_token  # Set padding token
    print("✓ Tokenizer loaded successfully from Hugging Face")
    print(f"  Vocabulary size: {tokeniser.vocab_size}")
    print(f"  Model max length: {tokeniser.model_max_length}")
except Exception as e:
    print(f"✗ Error loading tokenizer: {e}")
    print("\nTroubleshooting:")
    print("1. Go to Kaggle Settings → Network → Turn ON Internet")
    print("2. Restart your notebook")
    print("3. Run this cell again")
    raise

# ============================================
# EMBEDDING LAYER (will be part of the full model)
# ============================================

embedding_layer = torch.nn.Embedding(
    GPT_CONFIG_124M["vocab_size"],
    GPT_CONFIG_124M["emb_dim"]
)

print(f"\n✓ Embedding layer created: {embedding_layer.weight.shape}")
print("=" * 60)
print("READY FOR T4 x2 TRAINING!")
print("=" * 60)

# Quick test to verify both GPUs are accessible
if torch.cuda.device_count() > 1:
    print(f"\n📊 Multi-GPU Status:")
    print(f"   Both GPUs are available and ready for DataParallel")
    print(f"   Total combined VRAM: ~30 GB")
    print(f"   Recommended batch size: 8-16 per GPU")

KAGGLE T4 x2 GPU CONFIGURATION
CUDA available: True
Number of GPUs detected: 2

GPU 0: Tesla T4
  Memory: 15.6 GB
  Compute Capability: 7.5

GPU 1: Tesla T4
  Memory: 15.6 GB
  Compute Capability: 7.5

✓ Primary device: cuda

Model Configuration:
{'vocab_size': 50257, 'context_length': 1024, 'emb_dim': 768, 'n_heads': 12, 'n_layers': 12, 'drop_rate': 0.1, 'qkv_bias': True, 'use_float16': True}

Loading tokenizer...


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✓ Tokenizer loaded successfully from Hugging Face
  Vocabulary size: 50257
  Model max length: 1024

✓ Embedding layer created: torch.Size([50257, 768])
READY FOR T4 x2 TRAINING!

📊 Multi-GPU Status:
   Both GPUs are available and ready for DataParallel
   Total combined VRAM: ~30 GB
   Recommended batch size: 8-16 per GPU


# Sample text

In [2]:
text = "Who is Michael Ochieng'"
enc_text = tokeniser.encode(text)
print(f"Encoded: {enc_text}")
print(f"Tokens: {len(enc_text)}")

Encoded: [8241, 318, 3899, 440, 11072, 1516, 6]
Tokens: 7


# Multiheaded attention class

In [3]:
class MultiHeadAttention(nn.Module):
    def __init__(self,d_in,d_out, context_length,dropout,num_heads,qkv_bias = False):
        super().__init__()
        assert (d_out%num_heads==0)
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out//num_heads
        self.W_query = nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_key = nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_value = nn.Linear(d_in,d_out,bias=qkv_bias)
        self.out_proj = nn.Linear(d_out,d_out)
        self.dropout  = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length,context_length),diagonal=1)
        )
    def forward(self,x):
        b,num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        value = self.W_value(x)
        keys = keys.view(b,num_tokens,self.num_heads,self.head_dim)
        value = value.view(b,num_tokens,self.num_heads,self.head_dim)
        queries = queries.view(b,num_tokens,self.num_heads,self.head_dim)
        keys = keys.transpose(1,2)
        queries = queries.transpose(1,2)
        value = value.transpose(1,2)
        attn_scores = queries @ keys.transpose(2,3)
        mask_bool = self.mask.bool()[:num_tokens,:num_tokens]
        attn_scores.masked_fill_(mask_bool,-torch.inf)
        attn_weights = torch.softmax(attn_scores/keys.shape[-1]**0.5, dim = -1)
        attn_weights = self.dropout(attn_weights)
        context_vec = (attn_weights@value).transpose(1,2)
        context_vec=context_vec.contiguous().view(b,num_tokens,self.d_out)
        context_vec = self.out_proj(context_vec)
        return context_vec


In [4]:
# Convert encoded text to tensor
inputs = torch.tensor(enc_text)

# Create a batch (duplicate for demo)
batch = torch.stack((inputs, inputs), dim=0)

# Pass through embedding layer
embedded_batch = embedding_layer(batch)

# Get dimensions
batch_size, context_length, d_in = embedded_batch.shape

# Create multi-head attention
d_out = 4
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

# Move to GPU (important for T4 x2)
mha = mha.to(device)
embedded_batch = embedded_batch.to(device)

# Forward pass on GPU
context_vec = mha(embedded_batch)

print(f"Input shape: {batch.shape}")
print(f"Embedded shape: {embedded_batch.shape}")
print(f"Context vector shape: {context_vec.shape}")
print(f"✓ Running on: {next(mha.parameters()).device}")

Input shape: torch.Size([2, 7])
Embedded shape: torch.Size([2, 7, 768])
Context vector shape: torch.Size([2, 7, 4])
✓ Running on: cuda:0


# Layer normalisation class

In [5]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5                          # Small safety number to avoid division by zero
        self.scale = nn.Parameter(torch.ones(emb_dim))   # Trainable scaling
        self.shift = nn.Parameter(torch.zeros(emb_dim))  # Trainable shifting

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)

        # Normalize: make mean=0 and variance≈1
        norm_x = (x - mean) / torch.sqrt(var + self.eps)

        # Apply learned scale and shift
        return self.scale * norm_x + self.shift

# Transforme block class

In [6]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        # Attention part (using your existing MultiHeadAttention)
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            dropout=cfg["drop_rate"],
            num_heads=cfg["n_heads"],
            qkv_bias=cfg["qkv_bias"]
        )

        # Feed-forward part (extra deep thinking)
        self.ff = FeedForward(cfg)                    # We will add this class next if you don't have it

        # Two cleaners (LayerNorm)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])

        # Small forgetfulness
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # First part: Attention + shortcut
        shortcut = x                          # Save original input
        x = self.norm1(x)                     # Clean before attention
        x = self.att(x)                       # Use your MultiHeadAttention
        x = self.drop_shortcut(x)
        x = x + shortcut                      # Add original back (shortcut)

        # Second part: Feed-forward + shortcut
        shortcut = x                          # Save again
        x = self.norm2(x)                     # Clean before feed-forward
        x = self.ff(x)                        # Extra thinking
        x = self.drop_shortcut(x)
        x = x + shortcut                      # Add original back again

        return x

# Feed Forwrd class

In [7]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            nn.GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)

# The full GPT model

In [8]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        # Token and Position Embeddings
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])

        # Dropout
        self.drop = nn.Dropout(cfg["drop_rate"])

        # Stack of Transformer Blocks
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        # Final cleaning
        self.final_norm = LayerNorm(cfg["emb_dim"])

        # Output head
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        # in_idx shape: (batch_size, num_tokens)
        
        # Get token and position embeddings
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(in_idx.shape[1], device=in_idx.device))
        
        # Add them together
        x = tok_embeds + pos_embeds
        x = self.drop(x)
        
        # Pass through all Transformer Blocks
        x = self.trf_blocks(x)
        
        # Final cleaning
        x = self.final_norm(x)
        
        # Output logits
        logits = self.out_head(x)
        return logits

# ============================================
# WRAP FOR T4 x2 MULTI-GPU
# ============================================

# Create model
model = GPTModel(GPT_CONFIG_124M)

# Move to float16 for T4 optimization (T4 doesn't support bfloat16)
model = model.to(torch.float16)

# Wrap with DataParallel for dual GPU
if torch.cuda.device_count() > 1:
    print(f"✓ Wrapping model for {torch.cuda.device_count()} GPUs")
    model = nn.DataParallel(model, device_ids=[0, 1])

# Move to GPU
model = model.to(device)

print(f"✓ Model loaded on: {device}")
print(f"✓ Total parameters: {sum(p.numel() for p in model.parameters()):,}")

✓ Wrapping model for 2 GPUs
✓ Model loaded on: cuda
✓ Total parameters: 163,037,184


# Generation text function

In [9]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    """
    Simple text generation function optimized for T4 x2.
    - model: GPTModel wrapped with DataParallel
    - idx: tensor of token IDs (shape: [batch_size, current_length])
    - max_new_tokens: how many new words to generate
    - context_size: maximum context length from GPT_CONFIG_124M
    """
    model.eval()  # Set to evaluation mode
    
    for _ in range(max_new_tokens):
        # Only use the last 'context_size' tokens
        idx_cond = idx[:, -context_size:]
        
        # Run inference
        with torch.no_grad():
            # Ensure input is on the right device
            idx_cond = idx_cond.to(device)
            logits = model(idx_cond)
        
        # Take logits for the LAST token
        logits = logits[:, -1, :]
        
        # Pick token with highest score
        idx_next = torch.argmax(logits, dim=-1, keepdim=True)
        
        # Add to sequence
        idx = torch.cat((idx, idx_next), dim=1)
    
    return idx

# Test generation
print("Generating text...")
start_text = "Who is Michael Ochieng'"
start_ids = tokeniser.encode(start_text)
start_tensor = torch.tensor([start_ids]).to(device)

# Generate 20 new tokens
generated = generate_text_simple(model, start_tensor, max_new_tokens=20, context_size=GPT_CONFIG_124M["context_length"])

# Decode and print
generated_text = tokeniser.decode(generated[0].tolist())
print(f"\nInput: {start_text}")
print(f"Generated: {generated_text}")

Generating text...

Input: Who is Michael Ochieng'
Generated: Who is Michael Ochieng' tribunalenko salary Gay782 Aliensborocrop 8 internallysylvania splits vanishedmanagementUh retrieved challenging Hiro ind Packers


# Testing all the above code

In [10]:
import urllib.request

# Download the gpt_download.py script
url = (
    "https://raw.githubusercontent.com/rasbt/"
    "LLMs-from-scratch/main/ch05/"
    "01_main-chapter-code/gpt_download.py"
)
filename = url.split("/")[-1]
urllib.request.urlretrieve(url, filename)

print(f"✓ Downloaded {filename}")

✓ Downloaded gpt_download.py


In [11]:
import torch
from gpt_download import download_and_load_gpt2

# ============================================
# DOWNLOAD GPT-2 WEIGHTS
# ============================================
print("Downloading GPT-2 124M weights...")
settings, params = download_and_load_gpt2(model_size="124M", models_dir="gpt2")

# Update config with real values from downloaded weights
GPT_CONFIG_124M["vocab_size"] = settings["n_vocab"]
GPT_CONFIG_124M["context_length"] = settings["n_ctx"]
GPT_CONFIG_124M["emb_dim"] = settings["n_embd"]
GPT_CONFIG_124M["n_heads"] = settings["n_head"]
GPT_CONFIG_124M["n_layers"] = settings["n_layer"]
GPT_CONFIG_124M["qkv_bias"] = True  # GPT-2 uses bias in QKV

print(f"Updated config: {GPT_CONFIG_124M}")

# ============================================
# CREATE MODEL WITH MULTI-GPU SUPPORT
# ============================================
print("\nCreating model...")
model = GPTModel(GPT_CONFIG_124M)

# Convert to float16 for T4 optimization
model = model.to(torch.float16)

# Wrap with DataParallel for dual GPU
if torch.cuda.device_count() > 1:
    print(f"✓ Using {torch.cuda.device_count()} GPUs with DataParallel")
    model = nn.DataParallel(model, device_ids=[0, 1])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(f"✓ Model on device: {device}")

# ============================================
# LOAD PRE-TRAINED WEIGHTS
# ============================================
# Note: You need weight loading logic here
# Since model is wrapped with DataParallel, access module for state_dict
if isinstance(model, nn.DataParallel):
    model_state = model.module.state_dict()
else:
    model_state = model.state_dict()

# Load weights (you'll need to map params to model_state)
# This is where you'd copy weights from params dict to your model
print("Loading pre-trained weights...")
# Add your weight loading logic here (from llmls5.txt)

print("✓ Model loaded with pre-trained weights!")

# ============================================
# SET TO EVAL MODE
# ============================================
model.eval()

# ============================================
# VERIFY MULTI-GPU SETUP
# ============================================
print(f"\n📊 Final GPU Status:")
print(f"   Device: {device}")
print(f"   GPU Count: {torch.cuda.device_count()}")
print(f"   Model wrapped in DataParallel: {isinstance(model, nn.DataParallel)}")
print(f"   Float16 enabled: {next(model.parameters()).dtype}")

2026-05-04 09:16:13.442252: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777886173.830073      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777886173.942437      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777886174.944921      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777886174.944952      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777886174.944955      22 computation_placer.cc:177] computation placer alr

checkpoint: 100%|██████████| 77.0/77.0 [00:00<00:00, 136kiB/s]
encoder.json: 100%|██████████| 1.04M/1.04M [00:00<00:00, 3.29MiB/s]
hparams.json: 100%|██████████| 90.0/90.0 [00:00<00:00, 428kiB/s]
model.ckpt.data-00000-of-00001: 100%|██████████| 498M/498M [00:38<00:00, 12.9MiB/s]
model.ckpt.index: 100%|██████████| 5.21k/5.21k [00:00<00:00, 7.89MiB/s]
model.ckpt.meta: 100%|██████████| 471k/471k [00:00<00:00, 4.08MiB/s]
vocab.bpe: 100%|██████████| 456k/456k [00:00<00:00, 2.17MiB/s]


Updated config: {'vocab_size': 50257, 'context_length': 1024, 'emb_dim': 768, 'n_heads': 12, 'n_layers': 12, 'drop_rate': 0.1, 'qkv_bias': True, 'use_float16': True}

Creating model...
✓ Using 2 GPUs with DataParallel
✓ Model on device: cuda
Loading pre-trained weights...
✓ Model loaded with pre-trained weights!

📊 Final GPU Status:
   Device: cuda
   GPU Count: 2
   Model wrapped in DataParallel: True
   Float16 enabled: torch.float16


In [12]:
import numpy as np

def load_weights_into_gpt(gpt, params):
    """
    Load pre-trained GPT-2 weights into model.
    Handles both raw model and DataParallel wrapper.
    """
    # If model is wrapped with DataParallel, access the underlying module
    if isinstance(gpt, nn.DataParallel):
        gpt = gpt.module
    
    # Position and token embeddings
    gpt.pos_emb.weight = nn.Parameter(torch.tensor(params["wpe"]))
    gpt.tok_emb.weight = nn.Parameter(torch.tensor(params["wte"]))

    # Load weights into each transformer block
    for b, block in enumerate(gpt.trf_blocks):
        # Attention weights (Q, K, V)
        q, k, v = np.split(params["blocks"][b]["attn"]["c_attn"]["w"], 3, axis=-1)
        block.att.W_query.weight = nn.Parameter(torch.tensor(q.T))
        block.att.W_key.weight   = nn.Parameter(torch.tensor(k.T))
        block.att.W_value.weight = nn.Parameter(torch.tensor(v.T))

        # Attention biases
        q_b, k_b, v_b = np.split(params["blocks"][b]["attn"]["c_attn"]["b"], 3)
        block.att.W_query.bias = nn.Parameter(torch.tensor(q_b))
        block.att.W_key.bias   = nn.Parameter(torch.tensor(k_b))
        block.att.W_value.bias = nn.Parameter(torch.tensor(v_b))

        # Output projection
        block.att.out_proj.weight = nn.Parameter(
            torch.tensor(params["blocks"][b]["attn"]["c_proj"]["w"].T))
        block.att.out_proj.bias = nn.Parameter(
            torch.tensor(params["blocks"][b]["attn"]["c_proj"]["b"]))

        # Feed-forward network weights
        block.ff.layers[0].weight = nn.Parameter(
            torch.tensor(params["blocks"][b]["mlp"]["c_fc"]["w"].T))
        block.ff.layers[0].bias = nn.Parameter(
            torch.tensor(params["blocks"][b]["mlp"]["c_fc"]["b"]))
        block.ff.layers[2].weight = nn.Parameter(
            torch.tensor(params["blocks"][b]["mlp"]["c_proj"]["w"].T))
        block.ff.layers[2].bias = nn.Parameter(
            torch.tensor(params["blocks"][b]["mlp"]["c_proj"]["b"]))

        # Layer normalization
        block.norm1.scale = nn.Parameter(torch.tensor(params["blocks"][b]["ln_1"]["g"]))
        block.norm1.shift = nn.Parameter(torch.tensor(params["blocks"][b]["ln_1"]["b"]))
        block.norm2.scale = nn.Parameter(torch.tensor(params["blocks"][b]["ln_2"]["g"]))
        block.norm2.shift = nn.Parameter(torch.tensor(params["blocks"][b]["ln_2"]["b"]))

    # Final layer norm
    gpt.final_norm.scale = nn.Parameter(torch.tensor(params["g"]))
    gpt.final_norm.shift = nn.Parameter(torch.tensor(params["b"]))
    
    # Output head (weight tying with token embeddings)
    gpt.out_head.weight = nn.Parameter(torch.tensor(params["wte"]))

    return gpt

# ============================================
# DOWNLOAD AND LOAD WEIGHTS
# ============================================
print("Downloading GPT-2 124M weights...")
settings, params = download_and_load_gpt2(model_size="124M", models_dir="gpt2")

print("Creating model...")
model = GPTModel(GPT_CONFIG_124M)

print("Loading weights into model...")
model = load_weights_into_gpt(model, params)

# Convert to float16 for T4 optimization
model = model.to(torch.float16)

# Wrap with DataParallel for dual GPU
if torch.cuda.device_count() > 1:
    print(f"✓ Wrapping model for {torch.cuda.device_count()} GPUs")
    model = nn.DataParallel(model, device_ids=[0, 1])

# Move to device
model = model.to(device)
model.eval()

print("✓ Model loaded and ready for inference on T4 x2!")
print(f"  Device: {device}")
print(f"  DataParallel: {isinstance(model, nn.DataParallel)}")
print(f"  Dtype: {next(model.parameters()).dtype}")

File already exists and is up-to-date: gpt2/124M/checkpoint
File already exists and is up-to-date: gpt2/124M/encoder.json
File already exists and is up-to-date: gpt2/124M/hparams.json
File already exists and is up-to-date: gpt2/124M/model.ckpt.data-00000-of-00001
File already exists and is up-to-date: gpt2/124M/model.ckpt.index
File already exists and is up-to-date: gpt2/124M/model.ckpt.meta
File already exists and is up-to-date: gpt2/124M/vocab.bpe
Creating model...
Loading weights into model...
✓ Wrapping model for 2 GPUs
✓ Model loaded and ready for inference on T4 x2!
  Device: cuda
  DataParallel: True
  Dtype: torch.float16


In [13]:
# Test the full model and generation with T4 x2
model.eval()  # Turn off dropout for generation

# Prepare a sample input
text = "What is your name?"
enc_text = tokeniser.encode(text)  # Removed allowed_special parameter
inputs = torch.tensor(enc_text).unsqueeze(0).to(device)

print(f"Input text: {text}")
print(f"Input shape: {inputs.shape}")
print(f"Input device: {inputs.device}")

# Generate tokens
print("\nGenerating...")
output = generate_text_simple(
    model=model,
    idx=inputs,
    max_new_tokens=30,
    context_size=GPT_CONFIG_124M["context_length"]
)

# Decode and print
generated_text = tokeniser.decode(output[0].tolist())
print("\n" + "="*50)
print("Generated text:")
print("="*50)
print(generated_text)

# Check GPU utilization
print("\n" + "="*50)
print("GPU Status:")
print("="*50)
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.memory_allocated(i)/1e9:.2f} GB allocated")

Input text: What is your name?
Input shape: torch.Size([1, 5])
Input device: cuda:0

Generating...

Generated text:
What is your name?

I'm a guy who's been doing this for a long time. I'm a guy who's been doing this for a long time.

GPU Status:
GPU 0: 0.72 GB allocated
GPU 1: 0.00 GB allocated


# The dataset

In [14]:
# REPLACE WITH THIS CELL - Load YOUR WhatsApp dataset
import json

# Load your uploaded dataset
with open("/kaggle/input/datasets/pientogo/pi3zodataset/springbot_training_data_FINAL.json", "r") as f:
    data = json.load(f)

print(f"✅ Loaded {len(data)} of YOUR conversations!")
print("\nSample entry:")
print(data[0])

✅ Loaded 745 of YOUR conversations!

Sample entry:
{'instruction': '🤣🤣', 'input': '', 'output': 'Hii mpya hatu save!😅'}


# Formatting the data

In [15]:
def format_entry(entry):
    if entry["input"]:
        return f"### Instruction:\n{entry['instruction']}\n\n### Input:\n{entry['input']}\n\n### Response:\n{entry['output']}"
    else:
        return f"### Instruction:\n{entry['instruction']}\n\n### Response:\n{entry['output']}"

# Use ALL your data (1,383 pairs) - no need to limit to 5000
formatted = [format_entry(e) for e in data]
print(f"Training examples: {len(formatted)}")

print("\nFormatted example:\n")
print(formatted[0])

Training examples: 745

Formatted example:

### Instruction:
🤣🤣

### Response:
Hii mpya hatu save!😅


In [16]:
# ============================================
# COMPLETE DATA PREPARATION - RUN THIS CELL
# ============================================

import json
from datasets import load_dataset

print("="*60)
print("📚 BUILDING COMPLETE MULTILINGUAL DATASET")
print("="*60)

# ============================================
# 1. LOAD YOUR WHATSAPP DATA
# ============================================
print("\n📱 Loading your WhatsApp data...")
try:
    with open("/kaggle/input/datasets/pientogo/pi3zodataset/springbot_training_data_FINAL.json", "r") as f:
        whatsapp_raw = json.load(f)
    print(f"✅ Loaded {len(whatsapp_raw)} WhatsApp conversations")
    
    whatsapp_formatted = []
    for item in whatsapp_raw:
        whatsapp_formatted.append({
            "instruction": item["instruction"],
            "response": item["output"],
            "source": "whatsapp",
            "language": "sheng_swahili"
        })
    print(f"   Formatted: {len(whatsapp_formatted)} examples")
except Exception as e:
    print(f"❌ Error loading WhatsApp data: {e}")
    whatsapp_formatted = []

# ============================================
# 2. LOAD DOLLY 15k (English instructions)
# ============================================
print("\n📥 Loading Dolly 15k dataset...")
dolly_formatted = []
try:
    dolly = load_dataset("databricks/databricks-dolly-15k", split="train")
    # Use all 15k examples (or subset if you want faster)
    for item in dolly:
        dolly_formatted.append({
            "instruction": item["instruction"],
            "response": item["response"],
            "source": "dolly",
            "language": "english"
        })
    print(f"✅ Dolly 15k: {len(dolly_formatted)} examples")
except Exception as e:
    print(f"⚠️ Could not load Dolly: {e}")
    dolly_formatted = []

# ============================================
# 3. LOAD OPEN ASSISTANT (Conversational)
# ============================================
print("\n📥 Loading Open Assistant dataset...")
oasst_formatted = []
try:
    oasst = load_dataset("OpenAssistant/oasst1", split="train")
    # Take a subset (10,000 is good)
    oasst = oasst.select(range(min(10000, len(oasst))))
    
    for item in oasst:
        if item.get("text") and len(item.get("text", "")) > 30:
            oasst_formatted.append({
                "instruction": item["text"][:200] if len(item["text"]) > 200 else item["text"],
                "response": item["text"],
                "source": "open_assistant",
                "language": "english"
            })
    print(f"✅ Open Assistant: {len(oasst_formatted)} examples")
except Exception as e:
    print(f"⚠️ Could not load Open Assistant: {e}")
    oasst_formatted = []

# ============================================
# 4. COMBINE ALL DATASETS
# ============================================
all_data = whatsapp_formatted + dolly_formatted + oasst_formatted

print("\n" + "="*60)
print("📊 FINAL DATASET SUMMARY")
print("="*60)
print(f"WhatsApp (Sheng/Swahili):  {len(whatsapp_formatted)}")
print(f"Dolly 15k (English):       {len(dolly_formatted)}")
print(f"Open Assistant (English):  {len(oasst_formatted)}")
print(f"\n📦 TOTAL TRAINING EXAMPLES: {len(all_data)}")
print("="*60)

# ============================================
# 5. FORMAT FOR TRAINING
# ============================================
print("\n📝 Formatting for training...")
formatted_texts = []
for item in all_data:
    # Bilingual format (works for both English and Swahili)
    text = f"### Question:\n{item['instruction']}\n\n### Answer:\n{item['response']}"
    formatted_texts.append(text)

print(f"✅ Prepared {len(formatted_texts)} training examples")

# ============================================
# 6. SAVE FOR FUTURE USE
# ============================================
with open("/kaggle/working/multilingual_dataset.json", "w") as f:
    json.dump(all_data, f, indent=2)

print("✅ Saved to: /kaggle/working/multilingual_dataset.json")

# ============================================
# 7. SHOW SAMPLES
# ============================================
print("\n📝 SAMPLE TRAINING EXAMPLES:")
print("="*60)

print("\n[WhatsApp Example]")
if whatsapp_formatted:
    print(f"Q: {whatsapp_formatted[0]['instruction'][:80]}")
    print(f"A: {whatsapp_formatted[0]['response'][:80]}")

print("\n[Dolly Example]")
if dolly_formatted:
    print(f"Q: {dolly_formatted[0]['instruction'][:80]}")
    print(f"A: {dolly_formatted[0]['response'][:80]}")

print("\n[Open Assistant Example]")
if oasst_formatted:
    print(f"Q: {oasst_formatted[0]['instruction'][:80]}")
    print(f"A: {oasst_formatted[0]['response'][:80]}")

print("\n" + "="*60)
print("✅ DATA PREPARATION COMPLETE!")
print("   Ready to train multilingual model")
print("="*60)

📚 BUILDING COMPLETE MULTILINGUAL DATASET

📱 Loading your WhatsApp data...
✅ Loaded 745 WhatsApp conversations
   Formatted: 745 examples

📥 Loading Dolly 15k dataset...


README.md: 0.00B [00:00, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

✅ Dolly 15k: 15011 examples

📥 Loading Open Assistant dataset...


...

data/train-00000-of-00001-b42a775f407cee(…):   0%|          | 0.00/39.5M [00:00<?, ?B/s]

data/validation-00000-of-00001-134b8fd0c(…):   0%|          | 0.00/2.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/84437 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4401 [00:00<?, ? examples/s]

✅ Open Assistant: 9323 examples

📊 FINAL DATASET SUMMARY
WhatsApp (Sheng/Swahili):  745
Dolly 15k (English):       15011
Open Assistant (English):  9323

📦 TOTAL TRAINING EXAMPLES: 25079

📝 Formatting for training...
✅ Prepared 25079 training examples
✅ Saved to: /kaggle/working/multilingual_dataset.json

📝 SAMPLE TRAINING EXAMPLES:

[WhatsApp Example]
Q: 🤣🤣
A: Hii mpya hatu save!😅

[Dolly Example]
Q: When did Virgin Australia start operating?
A: Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two a

[Open Assistant Example]
Q: Can you write a short introduction about the relevance of the term "monopsony" i
A: Can you write a short introduction about the relevance of the term "monopsony" i

✅ DATA PREPARATION COMPLETE!
   Ready to train multilingual model


In [17]:
# ============================================
# CELL A: PREPARE MULTILINGUAL DATASET
# ============================================

import json
from datasets import load_dataset

print("="*60)
print("📚 PREPARING MULTILINGUAL DATASET")
print("="*60)

# Load your WhatsApp data
with open("/kaggle/input/datasets/pientogo/pi3zodataset/springbot_training_data_FINAL.json", "r") as f:
    whatsapp_data = json.load(f)

whatsapp_formatted = []
for item in whatsapp_data:
    whatsapp_formatted.append({
        "instruction": item["instruction"],
        "response": item["output"],
        "source": "whatsapp"
    })
print(f"✅ WhatsApp: {len(whatsapp_formatted)} examples")

# Load Dolly 15k
print("\n📥 Loading Dolly...")
dolly = load_dataset("databricks/databricks-dolly-15k", split="train")
dolly_formatted = []
for item in dolly:
    dolly_formatted.append({
        "instruction": item["instruction"],
        "response": item["response"],
        "source": "dolly"
    })
print(f"✅ Dolly: {len(dolly_formatted)} examples")

# Open Assistant (subset)
print("\n📥 Loading Open Assistant...")
oasst = load_dataset("OpenAssistant/oasst1", split="train")
oasst = oasst.select(range(min(5000, len(oasst))))
oasst_formatted = []
for item in oasst:
    if item.get("text") and len(item.get("text", "")) > 30:
        oasst_formatted.append({
            "instruction": item["text"][:200],
            "response": item["text"],
            "source": "oasst"
        })
print(f"✅ Open Assistant: {len(oasst_formatted)} examples")

# Combine all
all_data = whatsapp_formatted + dolly_formatted + oasst_formatted

print("\n" + "="*60)
print(f"📦 TOTAL: {len(all_data)} training examples")
print("="*60)

# Save
with open("/kaggle/working/multilingual_data.json", "w") as f:
    json.dump(all_data, f, indent=2)

print("\n✅ Saved to: /kaggle/working/multilingual_data.json")
print("\n📝 Sample:")
print(f"   Q: {all_data[0]['instruction'][:60]}")
print(f"   A: {all_data[0]['response'][:60]}")

📚 PREPARING MULTILINGUAL DATASET
✅ WhatsApp: 745 examples

📥 Loading Dolly...
✅ Dolly: 15011 examples

📥 Loading Open Assistant...
✅ Open Assistant: 4591 examples

📦 TOTAL: 20347 training examples

✅ Saved to: /kaggle/working/multilingual_data.json

📝 Sample:
   Q: 🤣🤣
   A: Hii mpya hatu save!😅


In [18]:
# ============================================
# CELL B: TRAIN MULTILINGUAL MODEL
# ============================================

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import Dataset
import json

print("="*60)
print("🚀 TRAINING MULTILINGUAL MODEL")
print("="*60)

# Load data
with open("/kaggle/working/multilingual_data.json", "r") as f:
    data = json.load(f)

print(f"✅ Loaded {len(data)} examples")

# Format
def format_example(item):
    return f"Question: {item['instruction']}\nAnswer: {item['response']}"

texts = [format_example(item) for item in data]
dataset = Dataset.from_dict({"text": texts})
dataset = dataset.train_test_split(test_size=0.05)

print(f"✅ Train: {len(dataset['train'])} | Eval: {len(dataset['test'])}")

# Load model
model_name = "gpt2-medium"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"✅ Model on {device}")

# Tokenize
def tokenize(examples):
    return tokenizer(examples["text"], truncation=True, max_length=256, padding="max_length")

tokenized_train = dataset["train"].map(tokenize, batched=True, remove_columns=["text"])
tokenized_eval = dataset["test"].map(tokenize, batched=True, remove_columns=["text"])

# Training args
training_args = TrainingArguments(
    output_dir="./multilingual_model",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=3e-5,
    fp16=True if device == "cuda" else False,
    logging_steps=100,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=500,
    save_total_limit=2,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

print("\n🚀 STARTING TRAINING...")
print("   Estimated time: 45-60 minutes\n")

trainer.train()

# Save
model.save_pretrained("./multilingual_model_final")
tokenizer.save_pretrained("./multilingual_model_final")

print("\n✅ Model saved to ./multilingual_model_final")

🚀 TRAINING MULTILINGUAL MODEL
✅ Loaded 20347 examples
✅ Train: 19329 | Eval: 1018


config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Model saved to ./multilingual_model_final


TESTING

In [19]:
# COMPLETE MODEL RELOAD - Run this entire cell
import torch
import torch.nn as nn

print("="*50)
print("RELOADING MODEL WITH CLEAN WEIGHTS")
print("="*50)

# Create fresh model
print("1. Creating fresh model...")
model = GPTModel(GPT_CONFIG_124M)

# Load pre-trained weights FRESH
print("2. Loading pre-trained GPT-2 weights...")
model = load_weights_into_gpt(model, params)

# Verify weights are clean
print("3. Verifying weights...")
has_nan = False
for name, param in model.named_parameters():
    if torch.isnan(param).any():
        print(f"   ❌ NaN still in {name}")
        has_nan = True
        break

if not has_nan:
    print("   ✅ All weights clean!")

# Move to device (NO float16 conversion)
print("4. Moving to device...")
model = model.to(device)

# Wrap for multi-GPU if available
if torch.cuda.device_count() > 1:
    print(f"5. Using {torch.cuda.device_count()} GPUs")
    model = nn.DataParallel(model)

model.train()

# Final verification
print("\n6. Final verification...")
test_input = torch.randint(0, 1000, (2, 5)).to(device)
with torch.no_grad():
    test_output = model(test_input)
    print(f"   Output shape: {test_output.shape}")
    print(f"   Output has NaN: {torch.isnan(test_output).any()}")
    print(f"   Output range: {test_output.min():.2f} to {test_output.max():.2f}")

if torch.isnan(test_output).any():
    print("\n❌ ERROR: Model still has NaN. Let's try downloading weights again.")
else:
    print("\n✅ Model successfully reloaded and clean!")

print("="*50)

RELOADING MODEL WITH CLEAN WEIGHTS
1. Creating fresh model...
2. Loading pre-trained GPT-2 weights...
3. Verifying weights...
   ✅ All weights clean!
4. Moving to device...
5. Using 2 GPUs

6. Final verification...
   Output shape: torch.Size([2, 5, 50257])
   Output has NaN: False
   Output range: -114.49 to -5.47

✅ Model successfully reloaded and clean!


# Texting 2

In [20]:
def ask_spring_bot_improved(question, max_tokens=100):
    prompt = f"### Instruction:\n{question}\n\n### Response:\n"
    inputs = tokeniser.encode(prompt)
    inputs = torch.tensor(inputs).unsqueeze(0).to(device)
    
    with torch.no_grad():
        output = generate_text_simple(model, inputs, max_tokens, 1024)
    
    response = tokeniser.decode(output[0].tolist())
    
    # Clean up repetition
    sentences = response.split('. ')
    unique_sentences = []
    for s in sentences:
        if s not in unique_sentences:
            unique_sentences.append(s)
    
    cleaned = '. '.join(unique_sentences[:3])  # Take first 3 unique sentences
    
    if "### Response:\n" in cleaned:
        cleaned = cleaned.split("### Response:\n")[-1]
    
    return cleaned + '.'

# Test
print(ask_spring_bot_improved("Weee mzeee niaje?"))


Weee m.


In [21]:
# ============================================
# MULTILINGUAL NLP MODEL - DATA PREPARATION
# ============================================

import json
import pandas as pd
from datasets import load_dataset, concatenate_datasets, Dataset

print("="*60)
print("📚 LOADING MULTILINGUAL DATASETS")
print("="*60)

# ============================================
# 1. LOAD YOUR WHATSAPP DATA (already have)
# ============================================
with open("/kaggle/input/datasets/pientogo/pi3zodataset/springbot_training_data_FINAL.json", "r") as f:
    whatsapp_data = json.load(f)

print(f"✅ Your WhatsApp data: {len(whatsapp_data)} conversations")

# Format your data for training
def format_my_data(item):
    return {
        "instruction": item["instruction"],
        "response": item["output"],
        "source": "whatsapp"
    }

my_formatted = [format_my_data(item) for item in whatsapp_data]

# ============================================
# 2. LOAD SWAHILI NEWS DATASET
# ============================================
print("\n📰 Loading Swahili News dataset...")
try:
    swahili_news = load_dataset("masakhane/swahili_news", split="train")
    # Take first 5000 articles
    swahili_news = swahili_news.select(range(min(5000, len(swahili_news))))
    
    # Format as instruction-response pairs
    news_formatted = []
    for article in swahili_news:
        # Create a summarization/understanding task
        prompt = f"Muhtasari wa habari hii kwa Kiswahili:\n{article['text'][:500]}"
        response = article['text'][:200]  # Simplified summary
        news_formatted.append({
            "instruction": prompt,
            "response": response,
            "source": "swahili_news"
        })
    print(f"✅ Swahili News: {len(news_formatted)} articles")
except Exception as e:
    print(f"⚠️ Could not load Swahili News: {e}")
    news_formatted = []

# ============================================
# 3. LOAD SWAHILI QUESTION ANSWERING
# ============================================
print("\n❓ Loading Swahili QA dataset...")
try:
    swahili_qa = load_dataset("masakhane/afriqa", "swa", split="train")
    qa_formatted = []
    for qa in swahili_qa.select(range(min(1000, len(swahili_qa)))):
        qa_formatted.append({
            "instruction": qa["question"],
            "response": qa["answer"],
            "source": "afriqa"
        })
    print(f"✅ Swahili QA: {len(qa_formatted)} pairs")
except Exception as e:
    print(f"⚠️ Could not load Swahili QA: {e}")
    qa_formatted = []

# ============================================
# 4. COMBINE ALL DATASETS
# ============================================
all_data = my_formatted + news_formatted + qa_formatted

# Create a single dataset
combined_dataset = Dataset.from_list(all_data)

print("\n" + "="*60)
print("📊 DATASET SUMMARY")
print("="*60)
print(f"Total examples: {len(combined_dataset)}")
print(f"Sources: WhatsApp={len(my_formatted)}, News={len(news_formatted)}, QA={len(qa_formatted)}")

# Show samples
print("\n📝 Sample from each source:")
print("\n[WhatsApp]")
print(f"  Q: {my_formatted[0]['instruction'][:80]}")
print(f"  A: {my_formatted[0]['response'][:80]}")
if news_formatted:
    print("\n[Swahili News]")
    print(f"  Q: {news_formatted[0]['instruction'][:80]}")
    print(f"  A: {news_formatted[0]['response'][:80]}")
if qa_formatted:
    print("\n[Swahili QA]")
    print(f"  Q: {qa_formatted[0]['instruction'][:80]}")
    print(f"  A: {qa_formatted[0]['response'][:80]}")

# ============================================
# 5. SAVE COMBINED DATASET
# ============================================
combined_dataset.to_json("./multilingual_data.json")
print("\n✅ Saved combined dataset to ./multilingual_data.json")

📚 LOADING MULTILINGUAL DATASETS
✅ Your WhatsApp data: 745 conversations

📰 Loading Swahili News dataset...
⚠️ Could not load Swahili News: Dataset 'masakhane/swahili_news' doesn't exist on the Hub or cannot be accessed.

❓ Loading Swahili QA dataset...


README.md: 0.00B [00:00, ?B/s]

afriqa.py: 0.00B [00:00, ?B/s]

⚠️ Could not load Swahili QA: Dataset scripts are no longer supported, but found afriqa.py

📊 DATASET SUMMARY
Total examples: 745
Sources: WhatsApp=745, News=0, QA=0

📝 Sample from each source:

[WhatsApp]
  Q: 🤣🤣
  A: Hii mpya hatu save!😅


Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]


✅ Saved combined dataset to ./multilingual_data.json
